# Image classification 

For classification purpose, the output layer needs to be a FC layer with its output the class number. 

ViT needs to append a FC layer (head) because its strucute was originally designed for NLP; for other CNN based model, can modify the last layer in-place.

Fine-tuning can:
1. Update the parameters of the whole model
2. Only update the last layer (or certain layers). Can make these parts require_grad=False.

Notes:
1. By default, pretrained PyTorch models (huggingface) are built considering input data (N,C,H,W) as the first layer or embedding layer. Model itself is built with weights only taking a single data (C,H, W) because no need to duplicated the parameters. For example, the ViT model used in this notebook handles batches in the model embedding layer.
2. PyTorch dataloader will prepare data in correct batch (N,C,H,W). Then you can write outputs=model(batch_data). It will feed each data into the "actual" model, calculate the score per data,and ouput the results in batch.
3. Then you can use criterion to calculate a single loss score per minibatch, and update the weights per minibatch (instead of the whole dataset).
4. PyTorch-Lightning handles batch inside the trainer() class.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import os

# Training
## Parameters

In [2]:
# Models to choose from [resnet, alexnet, vgg, squeezenet, densenet, inception]
model_name = 'vit_p32' #"vit_p32" #"resnet" #"squeezenet"

# Number of classes in the dataset
num_classes = 2

# Batch size during training
batch_size = 16

# Number of epochs to train for 
num_epochs = 1000

# Flag for feature extracting. When False, we finetune the whole model, when True we only update the reshaped layer params
feature_extract = True

# Number of GPUs available. Use 0 for CPU mode.
ngpu = 1

# Decide which device we want to run on
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
print(device)


round_info= 'round2'


cuda:0


## Training example

In [4]:
input_size=224
transform=transforms.Compose([transforms.Resize(input_size),
                              transforms.CenterCrop(input_size),
                              transforms.ToTensor(),
                              transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                             ])

In [5]:
from model_utils import myModels

# Initialize the model for this run
model_ft, input_size = myModels.initialize_model(model_name, num_classes, feature_extract, use_pretrained=True)


round_dir = '/lovelace/xiaoya/scientific_txt2image-main/classifier_checkpoints/'+round_info+'/'+model_name+'/'
model_ft = torch.load(round_dir+model_name+'_'+str(num_epochs)+'_epochs_classifier.pth')



Some weights of the model checkpoint at google/vit-base-patch32-224-in21k were not used when initializing ViTModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


# Evaluation example (vit_16x16)

In [6]:
import torchvision
from pipeline_utils import Evaluation

dataset = torchvision.datasets.ImageFolder(root="/lovelace/xiaoya/GenAI_dataset/xc_ring_1k_0_100_2nd_corrected", 
                                                   transform=transform)
eval_dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [7]:
#vit_p16 = torch.load(out_dir+model_name+'_'+str(num_epochs)+'_epochs_classifier.pth')
vit_p16_eval = Evaluation(model_ft, eval_dataloader, threshold=0.5, device=device)
vit_p16_eval.evaluate()

100%|███████████████████████████████████████████████████████| 13/13 [00:02<00:00,  5.37it/s]

avg acc: 0.8650000095367432, prec: 0.8613861203193665
